In [ ]:
import numpy as np
import pandas as pd

import tensorflow as tf

from sklearn import metrics

from tqdm import tqdm

import matplotlib.pyplot as plt

In [ ]:
training_data = pd.read_csv(
    'https://www.dropbox.com/s/z32q8nks8iqkiuv/waterDataTraining.csv?dl=1',
    index_col=0)
testing_data = pd.read_csv(
    'https://www.dropbox.com/s/3ptrkyisyks2us3/waterDataTesting.csv?dl=1',
    index_col=0)
#print(training_data)

In [ ]:
Xtrain = training_data.iloc[:,1:-1] # : significa todas as linhas, 1:-1 significa da segunda coluna até a ultima
Ytrain = np.array(training_data['EVENT']) # definição dos rotulos

Xtest = testing_data.iloc[:,1:-1]
Ytest = np.array(testing_data['EVENT'])

In [ ]:
Xtrain = Xtrain.fillna(method='ffill') # preenche os valores vazios com o valor anterior
Xtest = Xtest.fillna(method='ffill')

In [ ]:
# valores podem variar devagar ao longo do dia (por exemplo, a água fica um pouco mais quente à tarde e mais fria à noite)
# essas variações lentas são chamadas de tendência (trend)

window_trend = 24 * 60 # define uma janela deslizante de 24 horas

Xtrain_trend = Xtrain.rolling(window_trend, min_periods=1).mean() # para cada instante, calcula a média dos valores do último dia
Xtest_trend = Xtest.rolling(window_trend, min_periods=1).mean()

Xtrain = Xtrain - Xtrain_trend # subtrai a tendência de cada ponto
Xtest = Xtest - Xtest_trend

In [ ]:
val = round(Xtrain.shape[0]/2) # guarda o índice da metade do dataset

Xval = Xtrain[val:] # pega da metade até o final, vira o conjunto de validação (pra treinar internamente e ajustar hiperparametros)
Yval = Ytrain[val:]

Xtrain = Xtrain[:val]
Ytrain = Ytrain[:val]

In [ ]:
means = Xtrain.mean() # calcula as medias de cada coluna
stds = Xtrain.std() # calcula o desvio padrao de cada coluna

Xtrain = (Xtrain - means)/stds # normalizacao
Xval = (Xval - means)/stds
Xtest = (Xtest - means)/stds

# evita que variáveis com valores grandes dominem o aprendizado

O valor atual depende dos valores passados (ex: se a turbidez está aumentando há 10 minutos, pode indicar um evento (anomalia))

Um modelo que olha só uma linha isolada (um instante) não vê o histórico

**O objetivo é dar ao modelo uma memória do passado, transformando o dataset em sequências temporais**

In [ ]:
# transformando os dados em janelas temporais

window = 30 # o modelo vai olhar os últimos 30 registros (ex: minutos) para tentar prever o que acontece no instante atual

# cria uma lista com o primeiro registro repetido 29 vezes (se a janela for 30)
# transforma essa lista em um DataFrame
# junta esse pedaço “extra” no início do Xtrain
# assim Xtrain_ext fica maior do que Xtrain, com 29 linhas extras no começo
Xtrain_ext = pd.concat([pd.DataFrame([Xtrain.iloc[0]]*(window-1)), Xtrain])
Xval_ext = pd.concat([pd.DataFrame([Xval.iloc[0]]*(window-1)), Xval])
Xtest_ext = pd.concat([pd.DataFrame([Xtest.iloc[0]]*(window-1)), Xtest])

# cria uma matriz tridimensional (3D) para armazenar as janelas:
# Xtrain.shape[0] - Quantas amostras originais você tem
# window - Quantas observações por amostra (tamanho da janela)
# Xtrain.shape[1] - Quantas variáveis (features) cada amostra tem
Xtrain_3d = np.zeros((Xtrain.shape[0], window, Xtrain.shape[1]))
Xval_3d = np.zeros((Xval.shape[0], window, Xval.shape[1]))
Xtest_3d = np.zeros((Xtest.shape[0], window, Xtest.shape[1]))

# loop que cria as janelas
# pega 30 linhas consecutivas (a janela atual)
# centraliza os dados da janela (média zero)
# salva essa janela na posição i do array 3D
for i in tqdm(range(Xtrain.shape[0])):
    data = Xtrain_ext.iloc[i:i+window,:]
    Xtrain_3d[i] = (data - data.mean())

for i in tqdm(range(Xval.shape[0])):
    data = Xval_ext.iloc[i:i+window,:]
    Xval_3d[i] = (data - data.mean())

for i in tqdm(range(Xtest.shape[0])):
    data = Xtest_ext.iloc[i:i+window,:]
    Xtest_3d[i] = (data - data.mean())

In [ ]:
# divide o conjunto 3D em várias partes, uma para cada variável
# para que o modelo possa aprender separadamente os padrões de cada sensor (pH, turbidez, etc.) ao longo do tempo

n_features = Xtrain_3d.shape[2]

Xtrain_3d_split = [Xtrain_3d[:, :, i].reshape(Xtrain_3d.shape[0], Xtrain_3d.shape[1], 1) for i in range(n_features)]
Xval_3d_split = [Xval_3d[:, :, i].reshape(Xval_3d.shape[0], Xval_3d.shape[1], 1) for i in range(n_features)]
Xtest_3d_split = [Xtest_3d[:, :, i].reshape(Xtest_3d.shape[0], Xtest_3d.shape[1], 1) for i in range(n_features)]

In [ ]:
n_filters = 20 # modelo vai tentar aprender 20 tipos de padroes nos dados
kernel_size = 5 # "olhe sempre 5 minutos consecutivos para tentar encontrar um padrão"

epochs = 100 # quantas vezes o modelo vai passar pelo conjunto de dados
batch = 32 # treina em grupos de 32 janelas

In [ ]:
n_samples, n_window, n_features = Xtrain_3d.shape
n_filters = 13

for _ in range(1): # treinar 30 modelos

    model = tf.keras.Sequential() # cria um modelo sequencial no Keras, as camadas serão empilhadas uma após a outra em sequência

    # primeira camada Conv1D - aprende padrões temporais curtos dentro das janelas
    model.add(tf.keras.layers.Conv1D(filters=n_filters*n_features, kernel_size=kernel_size, activation='relu', input_shape=(n_window, n_features)))

    # segunda convolução 1D - aprofunda o aprendizado de padrões temporais mais complexos
    model.add(tf.keras.layers.Conv1D(filters=n_filters*n_features, kernel_size=kernel_size, activation='relu'))

    model.add(tf.keras.layers.GlobalMaxPooling1D()) # serve para resumir a sequência, pegando os padrões mais fortes detectados no tempo

    # essas 3 camadas aprendem combinações complexas dos padrões detectados pelas convoluções
    model.add(tf.keras.layers.Dense(n_filters*n_features,activation='relu'))
    model.add(tf.keras.layers.Dense(n_filters*n_features,activation='relu'))
    model.add(tf.keras.layers.Dense(n_filters*n_features,activation='relu'))

    # camada final com 1 neurônio, saída entre 0 e 1
    # é a probabilidade de acontecer o evento
    model.add(tf.keras.layers.Dense(1,activation='sigmoid'))

    # prepara o modelo para treinar
    # função de perda = binary cross-entropy
    # otimização = adam
    model.compile(loss="binary_crossentropy", optimizer='adam')

    # divide os índices das amostras entre positivas (onde há evento) e negativas
    # assim você pode sortear quantos exemplos quiser de cada classe
    indices_train_true = np.nonzero(Ytrain)[0]
    indices_train_false = np.nonzero(~Ytrain)[0]
    indices_val_true = np.nonzero(Yval)[0]
    indices_val_false = np.nonzero(~Yval)[0]

    for _ in tqdm(range(epochs)): # cada iteração treina o modelo em um subconjunto dos dados

        indices_train = np.concatenate((
            indices_train_true, # pega todas as amostras positivas
            np.random.choice(indices_train_false, # pega o mesmo número de negativas, escolhidas aleatoriamente
                            len(indices_train_true),
                            replace=False))) # não repete as mesmas amostras

        indices_val = np.concatenate(( # mesma ccoisa mas pro conjunto de validação
            indices_val_true,
            np.random.choice(indices_val_false,
                            len(indices_val_true),
                            replace=False)))

        model.fit(Xtrain_3d[indices_train], Ytrain[indices_train],
                validation_data=(Xval_3d[indices_val], Yval[indices_val]), # treina o modelo
                batch_size=batch, verbose=0)

    y_pred = model.predict(Xtest_3d) # pede pro modelo prever os valores do conjunto de teste

    print('F1:', np.max([metrics.f1_score(Ytest, y_pred>(th/100)) for th in range(101)])) # printa o maior valor de f1-score

# Monte carlo dropout

Normalmente o dropout é utilizado só durante o treino para evitar overfitting

Já o Monte Carlo Dropout proposto por Yarin Gal mantém o dropout ligado também na hora da previsão e faz várias passagens pelo modelo (com diferentes neurônios desligados aleatoriamente cada vez) vc consegue medir o quanto o modelo está incerto sobre aquela previsão

## resultado monte carlo dropout 1

In [ ]:
n_samples, n_window, n_features = Xtrain_3d.shape
n_filters = 13
drop_rate = 0.2 # 20% dos neurônios serão desligados
all_f1 = []
T = 20 # numero de vezes que vou repetir a previsão

# Função para previsão Monte Carlo Dropout em lotes menores (pra evitar estouro de memória)
def mc_dropout_predict_in_batches(model, X, T, batch_size):

    n_samples = X.shape[0]
    preds_all = []

    for t in range(T):

        preds_t = []

        for i in range(0, n_samples, batch_size): # preve em partes, de acordo com o meu batch_size, no caso faz 128 previsões por vez
            X_batch = X[i:i+batch_size] # pega uma fatia do subconjunto
            y_batch = model(X_batch, training=True)  # Dropout ativo
            preds_t.append(y_batch.numpy()) # transforma em um array

        preds_t = np.concatenate(preds_t, axis=0) # junta todos os batches da rodada em um vetor
        preds_all.append(preds_t)

        tf.keras.backend.clear_session()  # libera memória entre as iterações

    preds_all = np.stack(preds_all)

    mean = preds_all.mean(axis=0).ravel() # tira a média das previsoes
    std  = preds_all.std(axis=0).ravel() # calcula o desvio padrão

    return mean, std # retorna a média e o desvio padrão das previsões


for run in range(1): # treinar 30 modelos

    model = tf.keras.Sequential() # cria um modelo sequencial no Keras, as camadas serão empilhadas uma após a outra em sequência

    # primeira camada Conv1D - aprende padrões temporais curtos dentro das janelas
    model.add(tf.keras.layers.Conv1D(filters=n_filters*n_features, kernel_size=kernel_size, activation='relu', input_shape=(n_window, n_features)))
    model.add(tf.keras.layers.Dropout(drop_rate)) # mantem o dropout ativo

    # segunda convolução 1D - aprofunda o aprendizado de padrões temporais mais complexos
    model.add(tf.keras.layers.Conv1D(filters=n_filters*n_features, kernel_size=kernel_size, activation='relu'))
    model.add(tf.keras.layers.Dropout(drop_rate))

    model.add(tf.keras.layers.GlobalMaxPooling1D()) # serve para resumir a sequência, pegando os padrões mais fortes detectados no tempo

    # essas 3 camadas aprendem combinações complexas dos padrões detectados pelas convoluções
    model.add(tf.keras.layers.Dense(n_filters*n_features,activation='relu'))
    model.add(tf.keras.layers.Dropout(drop_rate))
    model.add(tf.keras.layers.Dense(n_filters*n_features,activation='relu'))
    model.add(tf.keras.layers.Dropout(drop_rate))
    model.add(tf.keras.layers.Dense(n_filters*n_features,activation='relu'))
    model.add(tf.keras.layers.Dropout(drop_rate))

    # camada final com 1 neurônio, saída entre 0 e 1
    # é a probabilidade de acontecer o evento
    model.add(tf.keras.layers.Dense(1,activation='sigmoid'))

    # prepara o modelo para treinar
    # função de perda = binary cross-entropy
    # otimização = adam
    model.compile(loss="binary_crossentropy", optimizer='adam')

    # divide os índices das amostras entre positivas (onde há evento) e negativas
    # assim você pode sortear quantos exemplos quiser de cada classe
    indices_train_true = np.nonzero(Ytrain)[0]
    indices_train_false = np.nonzero(~Ytrain)[0]
    indices_val_true = np.nonzero(Yval)[0]
    indices_val_false = np.nonzero(~Yval)[0]

    for _ in tqdm(range(epochs)): # cada iteração treina o modelo em um subconjunto dos dados

        indices_train = np.concatenate((
            indices_train_true, # pega todas as amostras positivas
            np.random.choice(indices_train_false, # pega o mesmo número de negativas, escolhidas aleatoriamente
                            len(indices_train_true),
                            replace=False))) # não repete as mesmas amostras

        indices_val = np.concatenate(( # mesma ccoisa mas pro conjunto de validação
            indices_val_true,
            np.random.choice(indices_val_false,
                            len(indices_val_true),
                            replace=False)))

        model.fit(Xtrain_3d[indices_train], Ytrain[indices_train],
                validation_data=(Xval_3d[indices_val], Yval[indices_val]), # treina o modelo
                batch_size=batch, verbose=0)

    # Monte Carlo Dropout com batches pequenos
    y_mean, y_std = mc_dropout_predict_in_batches(model, Xtest_3d, T=T, batch_size=64)

    # 10 amostras com menor desvio padrão
    idx_baixa = np.argsort(y_std)[:10]

    print("AMOSTRAS COM BAIXA INCERTEZA:")
    for i in idx_baixa:
        print(f"idx={i}, y_true={Ytest[i]}, y_mean={y_mean[i]:.4f}, y_std={y_std[i]:.6f}")

    # 10 amostras com maior desvio padrão
    idx_alta = np.argsort(y_std)[-10:]

    print("\nAMOSTRAS COM ALTA INCERTEZA:")
    for i in idx_alta:
        print(f"idx={i}, y_true={Ytest[i]}, y_mean={y_mean[i]:.4f}, y_std={y_std[i]:.6f}")

    # melhor F1-score
    best_f1 = np.max([metrics.f1_score(Ytest, y_mean > (th/100)) for th in range(101)])
    all_f1.append(best_f1)

    print(f"Modelo {run+1}/30 — F1: {best_f1:.4f} — Incerteza média: {y_std.mean():.4f}")

    tf.keras.backend.clear_session()

print(f"\nMédia do F1 em 30 execuções: {np.mean(all_f1):.4f} ± {np.std(all_f1):.4f}")

In [ ]:
model.summary()

# sub-redes separadas

Cria uma sub-rede convolucional independente para cada feature, Cada variável (pH, turbidez, condutividade...) entra por um Input separado,
passa pelas suas próprias duas convoluções + pooling,
e só depois todas as saídas são concatenadas.

In [ ]:
n_samples, n_window, n_features = Xtrain_3d.shape
n_filters = 20

for _ in range(1):

    visible = list()
    cnn = list()
    for i in range(n_features):

        visible.append(tf.keras.layers.Input(shape=(n_window,1)))

        cnn.append(tf.keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, activation='relu')(visible[i]))

        cnn[i] = tf.keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, activation='relu')(cnn[i])

        cnn[i] = tf.keras.layers.GlobalMaxPooling1D()(cnn[i])

    dense = tf.keras.layers.concatenate(cnn)

    dense = tf.keras.layers.Dense(n_features*n_filters,activation='relu')(dense)
    dense = tf.keras.layers.Dense(n_features*n_filters,activation='relu')(dense)
    dense = tf.keras.layers.Dense(n_features*n_filters,activation='relu')(dense)

    output = tf.keras.layers.Dense(1,activation='sigmoid')(dense)

    model = tf.keras.Model(inputs=visible, outputs=output)

    model.compile(loss="binary_crossentropy", optimizer='adam')

    indices_train_true = np.nonzero(Ytrain)[0]
    indices_train_false = np.nonzero(~Ytrain)[0]
    indices_val_true = np.nonzero(Yval)[0]
    indices_val_false = np.nonzero(~Yval)[0]
    for _ in tqdm(range(epochs)):

        indices_train = np.concatenate((
            indices_train_true,
            np.random.choice(indices_train_false,
                            len(indices_train_true),replace=False)))

        indices_val = np.concatenate((
            indices_val_true,
            np.random.choice(indices_val_false,
                            len(indices_val_true),replace=False)))

        Xtrain_under = []
        Xval_under = []
        for f in range(n_features):
            Xtrain_under.append(Xtrain_3d_split[f][indices_train])
            Xval_under.append(Xval_3d_split[f][indices_val])

        model.fit(Xtrain_under, Ytrain[indices_train], batch_size=batch,
                validation_data = (Xval_under, Yval[indices_val]),
                verbose=0)

    y_pred = model.predict(Xtest_3d_split)
    print('F1:', np.max([metrics.f1_score(Ytest, y_pred>(th/100)) for th in range(101)]))

## resultado com monte carlo dropout 2

In [ ]:
n_samples, n_window, n_features = Xtrain_3d.shape
n_filters = 20
drop_rate = 0.2
T = 20

def mc_dropout_predict_in_batches(model, X_list, T=20, batch_size=128):

    n_samples = X_list[0].shape[0]
    preds_all = []

    for t in range(T):
        preds_t = []
        for i in range(0, n_samples, batch_size):
            X_batch_list = [x[i:i+batch_size] for x in X_list]
            y_batch = model(X_batch_list, training=True)
            preds_t.append(y_batch.numpy())
        preds_t = np.concatenate(preds_t, axis=0)
        preds_all.append(preds_t)
        tf.keras.backend.clear_session()

    preds_all = np.stack(preds_all)
    mean = preds_all.mean(axis=0).ravel()
    std  = preds_all.std(axis=0).ravel()
    return mean, std

for run in range(1):

    visible = []
    cnn = []

    for i in range(n_features):

        visible.append(tf.keras.layers.Input(shape=(n_window, 1)))


        c = tf.keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size,
                                   activation='relu')(visible[i])
        c = tf.keras.layers.Dropout(drop_rate)(c)
        c = tf.keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size,
                                   activation='relu')(c)
        c = tf.keras.layers.Dropout(drop_rate)(c)

        c = tf.keras.layers.GlobalMaxPooling1D()(c)
        cnn.append(c)

    # concatena todas as features (saídas das sub-CNNs)
    dense = tf.keras.layers.concatenate(cnn)

    # densas finais com Dropout
    dense = tf.keras.layers.Dense(n_features*n_filters, activation='relu')(dense)
    dense = tf.keras.layers.Dropout(drop_rate)(dense)
    dense = tf.keras.layers.Dense(n_features*n_filters, activation='relu')(dense)
    dense = tf.keras.layers.Dropout(drop_rate)(dense)
    dense = tf.keras.layers.Dense(n_features*n_filters, activation='relu')(dense)
    dense = tf.keras.layers.Dropout(drop_rate)(dense)

    # saída probabilística
    output = tf.keras.layers.Dense(1, activation='sigmoid')(dense)

    # monta o modelo
    model = tf.keras.Model(inputs=visible, outputs=output)
    model.compile(loss="binary_crossentropy", optimizer='adam')


    indices_train_true = np.nonzero(Ytrain)[0]
    indices_train_false = np.nonzero(~Ytrain)[0]
    indices_val_true = np.nonzero(Yval)[0]
    indices_val_false = np.nonzero(~Yval)[0]
    for _ in tqdm(range(epochs)):

        indices_train = np.concatenate((
            indices_train_true,
            np.random.choice(indices_train_false,
                            len(indices_train_true),replace=False)))

        indices_val = np.concatenate((
            indices_val_true,
            np.random.choice(indices_val_false,
                            len(indices_val_true),replace=False)))

        Xtrain_under = []
        Xval_under = []
        for f in range(n_features):
            Xtrain_under.append(Xtrain_3d_split[f][indices_train])
            Xval_under.append(Xval_3d_split[f][indices_val])

        model.fit(Xtrain_under, Ytrain[indices_train], batch_size=batch,
                validation_data = (Xval_under, Yval[indices_val]),
                verbose=0)

    # faz T previsões com dropout ativo
    y_mean, y_std = mc_dropout_predict_in_batches(model, Xtest_3d_split,
                                                  T=T, batch_size=64)

    # avalia o melhor F1-score
    best_f1 = np.max([
        metrics.f1_score(Ytest, y_mean > (th/100))
        for th in range(101)
    ])

    print(f"Modelo {run+1}/30 — F1: {best_f1:.4f} — Incerteza média: {y_std.mean():.4f}")

    tf.keras.backend.clear_session()

In [ ]:
from google.colab import runtime
runtime.unassign()

# Parte 2 - MODELO COM BETA NLL LOSS

In [ ]:
import tensorflow_probability as tfp
import numpy as np
from sklearn import metrics

In [ ]:
# ajustando o formato para o modelo beta
Ytrain = Ytrain.reshape(-1, 1)
Yval   = Yval.reshape(-1, 1)
Ytest  = Ytest.reshape(-1, 1)

In [ ]:
def beta_nll_loss(y_true, y_pred):
    alpha = y_pred[:, 0:1]
    beta  = y_pred[:, 1:2]

    dist = tfp.distributions.Beta(alpha, beta)
    loss = -dist.log_prob(y_true)

    return tf.reduce_mean(loss)

In [ ]:
Ytrain_original = (Ytrain > 0.5).astype(int)
Yval_original   = (Yval > 0.5).astype(int)
Ytest_original  = (Ytest > 0.5).astype(int)

eps = 1e-4

Ytrain = Ytrain * (1 - 2*eps) + eps
Yval   = Yval * (1 - 2*eps) + eps
Ytest  = Ytest * (1 - 2*eps) + eps

In [ ]:
all_f1_beta = []
n_samples, n_window, n_features = Xtrain_3d.shape
drop_rate = 0.2

for run in range(1):

    inputs = tf.keras.Input(shape=(n_window, n_features))

    x = tf.keras.layers.Conv1D(filters=n_filters*n_features, kernel_size=kernel_size, activation='relu')(inputs)
    x = tf.keras.layers.Dropout(drop_rate)(x)

    x = tf.keras.layers.Conv1D(filters=n_filters*n_features, kernel_size=kernel_size, activation='relu')(x)
    x = tf.keras.layers.Dropout(drop_rate)(x)

    x = tf.keras.layers.GlobalMaxPooling1D()(x)

    x = tf.keras.layers.Dense(n_filters*n_features, activation='relu')(x)
    x = tf.keras.layers.Dropout(drop_rate)(x)
    x = tf.keras.layers.Dense(n_filters*n_features, activation='relu')(x)
    x = tf.keras.layers.Dropout(drop_rate)(x)
    x = tf.keras.layers.Dense(n_filters*n_features, activation='relu')(x)
    x = tf.keras.layers.Dropout(drop_rate)(x)

    # SAÍDAS BETA

    alpha = tf.keras.layers.Dense(1, activation='softplus')(x)
    beta  = tf.keras.layers.Dense(1, activation='softplus')(x)

    alpha = tf.keras.layers.Lambda(lambda x: x + 1e-6)(alpha)
    beta  = tf.keras.layers.Lambda(lambda x: x + 1e-6)(beta)

    outputs = tf.keras.layers.Concatenate()([alpha, beta])

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer='adam', loss=beta_nll_loss)

    # ===== TREINO =====
    indices_train_true = np.nonzero(Ytrain_original.flatten())[0]
    indices_train_false = np.nonzero(~Ytrain_original.flatten().astype(bool))[0]

    indices_val_true = np.nonzero(Yval_original.flatten())[0]
    indices_val_false = np.nonzero(~Yval_original.flatten().astype(bool))[0]

    for _ in tqdm(range(epochs)):

        indices_train = np.concatenate((
            indices_train_true,
            np.random.choice(indices_train_false, len(indices_train_true), replace=False)
        ))

        indices_val = np.concatenate((
            indices_val_true,
            np.random.choice(indices_val_false, len(indices_val_true), replace=False)
        ))

        model.fit(
            Xtrain_3d[indices_train],
            Ytrain[indices_train],
            validation_data=(Xval_3d[indices_val], Yval[indices_val]),
            batch_size=batch,
            verbose=0
        )

    # ===== INFERÊNCIA =====
    pred = model.predict(Xtest_3d)

    alpha_pred = pred[:, 0:1]
    beta_pred  = pred[:, 1:2]

    y_pred_prob = alpha_pred / (alpha_pred + beta_pred)

    eps = 1e-8

    uncertainty = (alpha_pred * beta_pred) / (
        (alpha_pred + beta_pred + eps)**2 * (alpha_pred + beta_pred + 1)
    )

    # ===== F1 =====
    best_f1 = np.max([
        metrics.f1_score(Ytest_original, y_pred_prob > (th/100))
        for th in range(101)
    ])

    all_f1_beta.append(best_f1)

    # ===== AMOSTRAS COM BAIXA INCERTEZA =====
    idx_baixa = np.argsort(uncertainty.ravel())[:10]

    print("AMOSTRAS COM BAIXA INCERTEZA:")
    for i in idx_baixa:
        print(f"idx={i}, y_true={Ytest_original[i][0]}, "
              f"y_pred={y_pred_prob[i][0]:.4f}, "
              f"uncertainty={uncertainty[i][0]:.6f}")

    # ===== AMOSTRAS COM ALTA INCERTEZA =====
    idx_alta = np.argsort(uncertainty.ravel())[-10:]

    print("\nAMOSTRAS COM ALTA INCERTEZA:")
    for i in idx_alta:
        print(f"idx={i}, y_true={Ytest_original[i][0]}, "
              f"y_pred={y_pred_prob[i][0]:.4f}, "
              f"uncertainty={uncertainty[i][0]:.6f}")

    print(f"Modelo {run+1} — F1: {best_f1:.4f} — Incerteza média: {uncertainty.mean():.4f}")

## MODELO COM BETA NLL LOSS 2

In [ ]:
n_samples, n_window, n_features = Xtrain_3d.shape
n_filters = 20
drop_rate = 0.2
all_f1_beta = []

for run in range(1):

    visible = []
    cnn = []

    for i in range(n_features):

        visible.append(tf.keras.layers.Input(shape=(n_window, 1)))


        c = tf.keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size,
                                   activation='relu')(visible[i])
        c = tf.keras.layers.Dropout(drop_rate)(c)
        c = tf.keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size,
                                   activation='relu')(c)
        c = tf.keras.layers.Dropout(drop_rate)(c)

        c = tf.keras.layers.GlobalMaxPooling1D()(c)
        cnn.append(c)

    # concatena todas as features (saídas das sub-CNNs)
    dense = tf.keras.layers.concatenate(cnn)

    # densas finais com Dropout
    dense = tf.keras.layers.Dense(n_features*n_filters, activation='relu')(dense)
    dense = tf.keras.layers.Dropout(drop_rate)(dense)
    dense = tf.keras.layers.Dense(n_features*n_filters, activation='relu')(dense)
    dense = tf.keras.layers.Dropout(drop_rate)(dense)
    dense = tf.keras.layers.Dense(n_features*n_filters, activation='relu')(dense)
    dense = tf.keras.layers.Dropout(drop_rate)(dense)

    # saída beta
    alpha = tf.keras.layers.Dense(1, activation='softplus')(dense)
    beta  = tf.keras.layers.Dense(1, activation='softplus')(dense)

    alpha = tf.keras.layers.Lambda(lambda x: x + 1e-6)(alpha)
    beta  = tf.keras.layers.Lambda(lambda x: x + 1e-6)(beta)

    outputs = tf.keras.layers.Concatenate()([alpha, beta])

    model = tf.keras.Model(inputs=visible, outputs=outputs)
    model.compile(optimizer='adam', loss=beta_nll_loss)

    # balanceamento
    indices_train_true = np.nonzero(Ytrain_original.flatten())[0]
    indices_train_false = np.nonzero(~Ytrain_original.flatten().astype(bool))[0]

    indices_val_true = np.nonzero(Yval_original.flatten())[0]
    indices_val_false = np.nonzero(~Yval_original.flatten().astype(bool))[0]

    # treino
    for _ in tqdm(range(epochs)):

        indices_train = np.concatenate((
            indices_train_true,
            np.random.choice(indices_train_false,
                            len(indices_train_true),replace=False)))

        indices_val = np.concatenate((
            indices_val_true,
            np.random.choice(indices_val_false,
                            len(indices_val_true),replace=False)))

        Xtrain_under = []
        Xval_under = []

        for f in range(n_features):
            Xtrain_under.append(Xtrain_3d_split[f][indices_train])
            Xval_under.append(Xval_3d_split[f][indices_val])

        model.fit(Xtrain_under, Ytrain[indices_train], batch_size=batch,
                validation_data = (Xval_under, Yval[indices_val]),
                verbose=0)

    # ===== INFERÊNCIA =====
    pred = model.predict(Xtest_3d_split)

    alpha_pred = pred[:, 0:1]
    beta_pred  = pred[:, 1:2]

    y_pred_prob = alpha_pred / (alpha_pred + beta_pred)

    # ===== ACURÁCIA POR CLASSE =====

    # transforma probabilidade em classe
    y_pred = (y_pred_prob > 0.5).astype(int)

    # deixa em formato 1D
    y_true = Ytest_original.ravel()
    y_pred = y_pred.ravel()

    # acurácia da classe 0
    acc_classe_0 = np.mean(y_pred[y_true == 0] == 0)

    # acurácia da classe 1
    acc_classe_1 = np.mean(y_pred[y_true == 1] == 1)

    print("\n===== ACURÁCIA POR CLASSE =====")
    print(f"Acurácia classe 0 - Normal: {acc_classe_0:.4f}")
    print(f"Acurácia classe 1 - Evento: {acc_classe_1:.4f}")

    eps = 1e-8

    uncertainty = (alpha_pred * beta_pred) / (
        (alpha_pred + beta_pred + eps)**2 * (alpha_pred + beta_pred + 1)
    )

    # ===== INCERTEZA POR CLASSE =====

    uncertainty_1d = uncertainty.ravel()
    y_true = Ytest_original.ravel()

    incerteza_classe_0 = np.mean(uncertainty_1d[y_true == 0])
    incerteza_classe_1 = np.mean(uncertainty_1d[y_true == 1])

    print("\n===== INCERTEZA MÉDIA POR CLASSE =====")
    print(f"Incerteza classe 0 - Normal: {incerteza_classe_0:.6f}")
    print(f"Incerteza classe 1 - Evento: {incerteza_classe_1:.6f}")

    # ===== AMOSTRAS COM BAIXA INCERTEZA =====
    idx_baixa = np.argsort(uncertainty.ravel())[:10]

    print("\nAMOSTRAS COM BAIXA INCERTEZA:")
    for i in idx_baixa:
        print(f"idx={i}, "
              f"y_true={Ytest_original[i][0]}, "
              f"y_pred={y_pred_prob[i][0]:.4f}, "
              f"uncertainty={uncertainty[i][0]:.6f}")

    # ===== AMOSTRAS COM ALTA INCERTEZA =====
    idx_alta = np.argsort(uncertainty.ravel())[-10:]

    print("\nAMOSTRAS COM ALTA INCERTEZA:")
    for i in idx_alta:
        print(f"idx={i}, "
              f"y_true={Ytest_original[i][0]}, "
              f"y_pred={y_pred_prob[i][0]:.4f}, "
              f"uncertainty={uncertainty[i][0]:.6f}")

    # ===== F1 =====
    best_f1 = np.max([
        metrics.f1_score(Ytest_original, y_pred_prob > (th/100))
        for th in range(101)
    ])

    all_f1_beta.append(best_f1)

    print(f"Modelo {run+1} — F1: {best_f1:.4f} — Incerteza média: {uncertainty.mean():.4f}")